In [1]:
import pandas as pd
import numpy as np
from surprise import SVD, Dataset, Reader
from surprise.model_selection import train_test_split, cross_validate
from surprise import accuracy

print("All imports successful!")

All imports successful!


In [2]:
df = pd.read_csv('../data/processed/interactions_filtered.csv')
print(df.shape)
df.head()

(40879, 10)


,user_id,book_id,review_id,is_read,rating,review_text_incomplete,date_added,date_updated,read_at,started_at
0,8842281e1d1347389f2ab93d60773d4d,23310161,f4b4b050f4be00e9283c92a814af2670,True,4,Fun sequel to the original.,Tue Nov 17 11:37:35 -0800 2015,Tue Nov 17 11:38:05 -0800 2015,NaN,NaN
1,8842281e1d1347389f2ab93d60773d4d,817720,75fd46041466ceb406b7fd69b089b9c5,True,5,NaN,Wed May 20 21:29:23 -0700 2015,Wed May 20 21:29:23 -0700 2015,NaN,NaN
2,8842281e1d1347389f2ab93d60773d4d,1969280,5809d5592ee32745e048a9c67ac27100,True,5,NaN,Sat Nov 08 08:56:58 -0800 2014,Wed Dec 17 00:37:25 -0800 2014,NaN,NaN
3,8842281e1d1347389f2ab93d60773d4d,17290220,22d424a2b0057b18fb6ecf017af7be92,True,5,One of my favorite books to read to my 5 year ...,Sat Nov 08 08:54:03 -0800 2014,Wed Jan 25 13:56:12 -0800 2017,Tue Jan 24 00:00:00 -0800 2017,NaN
4,8842281e1d1347389f2ab93d60773d4d,1027760,0c8a75acde799d70696f4aecf2d611de,True,4,NaN,Tue Mar 18 22:23:03 -0700 2014,Wed Mar 22 11:47:31 -0700 2017,NaN,NaN


In [3]:
# Tell surprise the rating scale
reader = Reader(rating_scale=(1, 5))

# Load only the three columns surprise needs, in this exact order
data = Dataset.load_from_df(df[['user_id', 'book_id', 'rating']], reader)

print(type(data))
print("Data loaded into surprise format successfully!")

<class 'surprise.dataset.DatasetAutoFolds'>
Data loaded into surprise format successfully!


In [4]:
# Split into 80% train, 20% test
# random_state=42 makes it reproducible
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

print(f"Training interactions: {trainset.n_ratings:,}")
print(f"Test interactions:     {len(testset):,}")
print(f"Unique users in train: {trainset.n_users:,}")
print(f"Unique items in train: {trainset.n_items:,}")

Training interactions: 32,703
Test interactions:     8,176
Unique users in train: 1,906
Unique items in train: 1,894


In [5]:
# Define the model with default hyperparameters
model = SVD(
    n_factors=100,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42,
    verbose=True  # prints loss each epoch so you can watch it learn
)

# Train on the training set
model.fit(trainset)

Processing epoch 0
Processing epoch 1
Processing epoch 2
Processing epoch 3
Processing epoch 4
Processing epoch 5
Processing epoch 6
Processing epoch 7
Processing epoch 8
Processing epoch 9
Processing epoch 10
Processing epoch 11
Processing epoch 12
Processing epoch 13
Processing epoch 14
Processing epoch 15
Processing epoch 16
Processing epoch 17
Processing epoch 18
Processing epoch 19


In [6]:
# Generate predictions on the test set
predictions = model.test(testset)

# Calculate RMSE
rmse = accuracy.rmse(predictions)
print(f"\nBaseline RMSE to beat: 0.9060")
print(f"SVD RMSE:              {rmse:.4f}")
print(f"Improvement:           {0.9060 - rmse:.4f}")

RMSE: 0.7992

Baseline RMSE to beat: 0.9060
SVD RMSE:              0.7992
Improvement:           0.1068


In [7]:
def get_recommendations(user_id, n=10):
    # All books in the training set
    all_book_ids = df['book_id'].unique()
    
    # Books this user has already rated
    rated_books = df[df['user_id'] == user_id]['book_id'].values
    
    # Books they haven't rated yet
    unrated_books = [b for b in all_book_ids if b not in rated_books]
    
    # Predict rating for each unrated book
    predictions = [model.predict(user_id, book_id) for book_id in unrated_books]
    
    # Sort by estimated rating descending
    predictions.sort(key=lambda x: x.est, reverse=True)
    
    # Return top N
    top_n = predictions[:n]
    print(f"Top {n} recommendations for user {user_id[:8]}...:\n")
    for i, pred in enumerate(top_n, 1):
        print(f"{i:2}. Book {pred.iid} — predicted rating: {pred.est:.2f}")

# Pick any user from the dataset
sample_user = df['user_id'].iloc[0]
get_recommendations(sample_user, n=10)

Top 10 recommendations for user 8842281e...:

 1. Book 44186 — predicted rating: 5.00
 2. Book 47693 — predicted rating: 5.00
 3. Book 144974 — predicted rating: 5.00
 4. Book 11387515 — predicted rating: 5.00
 5. Book 443103 — predicted rating: 5.00
 6. Book 25311520 — predicted rating: 5.00
 7. Book 24612624 — predicted rating: 5.00
 8. Book 16101018 — predicted rating: 5.00
 9. Book 10574666 — predicted rating: 5.00
10. Book 37186 — predicted rating: 5.00


In [8]:
# Only consider books with enough ratings to be reliable
popular_books = df.groupby('book_id')['rating'].count()
reliable_books = popular_books[popular_books >= 10].index

def get_recommendations_v2(user_id, n=10):
    # Books this user has already rated
    rated_books = set(df[df['user_id'] == user_id]['book_id'].values)
    
    # Only predict on reliable, unrated books
    candidate_books = [b for b in reliable_books if b not in rated_books]
    
    # Predict and sort
    predictions = [model.predict(user_id, book_id) for book_id in candidate_books]
    predictions.sort(key=lambda x: x.est, reverse=True)
    
    top_n = predictions[:n]
    print(f"Top {n} recommendations for user {user_id[:8]}...:\n")
    for i, pred in enumerate(top_n, 1):
        print(f"{i:2}. Book {pred.iid} — predicted rating: {pred.est:.2f}")

get_recommendations_v2(sample_user, n=10)

Top 10 recommendations for user 8842281e...:

 1. Book 378 — predicted rating: 5.00
 2. Book 6328 — predicted rating: 5.00
 3. Book 7772 — predicted rating: 5.00
 4. Book 7784 — predicted rating: 5.00
 5. Book 7926 — predicted rating: 5.00
 6. Book 8073 — predicted rating: 5.00
 7. Book 11903 — predicted rating: 5.00
 8. Book 14118 — predicted rating: 5.00
 9. Book 17490 — predicted rating: 5.00
10. Book 19330 — predicted rating: 5.00


In [9]:
user_ratings = df[df['user_id'] == sample_user]['rating']
print(f"Total ratings:   {len(user_ratings)}")
print(f"Average rating:  {user_ratings.mean():.2f}")
print(f"Rating breakdown:\n{user_ratings.value_counts().sort_index()}")

Total ratings:   32
Average rating:  4.91
Rating breakdown:
rating
4     3
5    29
Name: count, dtype: int64


In [10]:
# Try a few different users
for user_id in df['user_id'].unique()[1:5]:
    user_avg = df[df['user_id'] == user_id]['rating'].mean()
    user_count = df[df['user_id'] == user_id]['rating'].count()
    print(f"\nUser {user_id[:8]}... | {user_count} ratings | avg: {user_avg:.2f}")
    get_recommendations_v2(user_id, n=5)
    print("-" * 50)


User 72fb0d00... | 30 ratings | avg: 4.87
Top 5 recommendations for user 72fb0d00...:

 1. Book 3579 — predicted rating: 5.00
 2. Book 7784 — predicted rating: 5.00
 3. Book 17490 — predicted rating: 5.00
 4. Book 19330 — predicted rating: 5.00
 5. Book 24335 — predicted rating: 5.00
--------------------------------------------------

User 06316bec... | 7 ratings | avg: 4.57
Top 5 recommendations for user 06316bec...:

 1. Book 240130 — predicted rating: 4.83
 2. Book 11387515 — predicted rating: 4.78
 3. Book 3579 — predicted rating: 4.76
 4. Book 310259 — predicted rating: 4.71
 5. Book 10508431 — predicted rating: 4.71
--------------------------------------------------

User 1711b2a4... | 6 ratings | avg: 3.50
Top 5 recommendations for user 1711b2a4...:

 1. Book 79626 — predicted rating: 4.42
 2. Book 56728 — predicted rating: 4.37
 3. Book 13083239 — predicted rating: 4.32
 4. Book 770051 — predicted rating: 4.32
 5. Book 240007 — predicted rating: 4.31
--------------------------

In [11]:
# 5-fold cross validation gives a more reliable performance estimate
# than a single train/test split
cv_results = cross_validate(
    SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42),
    data,
    measures=['RMSE', 'MAE'],
    cv=5,
    verbose=True
)

print(f"\nMean RMSE across 5 folds: {cv_results['test_rmse'].mean():.4f}")
print(f"Std RMSE:                 {cv_results['test_rmse'].std():.4f}")
print(f"Mean MAE:                 {cv_results['test_mae'].mean():.4f}")
print(f"\nBaseline RMSE:            0.9060")
print(f"SVD RMSE (single split):  0.7992")

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.7855  0.7916  0.8006  0.7887  0.7922  0.7917  0.0050  
MAE (testset)     0.6233  0.6286  0.6330  0.6262  0.6259  0.6274  0.0032  
Fit time          0.34    0.33    0.29    0.29    0.28    0.30    0.02    
Test time         0.05    0.04    0.04    0.04    0.04    0.04    0.00    

Mean RMSE across 5 folds: 0.7917
Std RMSE:                 0.0050
Mean MAE:                 0.6274

Baseline RMSE:            0.9060
SVD RMSE (single split):  0.7992
